# Runner — Motor GES/CAEC lite

Interactive front end for the engine. It follows the pipeline: **extract** (`src/extractor.py`) → **transform** (`src/calculo.py`, pure) → **load** (`src/loader.py`), wired by `src/motor.py`, which is also the CLI.

| § | What | Database |
|---|---|---|
| 1 | Run the tests | — |
| 2 | **Extract**: the seed data, and what the system knew at a given moment | in memory |
| 3 | **Transform**: the pure calculation, step by step | in memory |
| 4 | **Load**: full pipeline, saved result, idempotence | in memory |
| 5 | Recalculation: the four scenarios that justify the project | in memory |
| 6 | Excel ingestion | `data/runner.db`, recreated |
| 7 | **Your database**: operate the engine through the CLI | `data/motor.db`, kept |

- **Kernel:** the repo's `.venv` (Python 3.12), with `pip install -r requirements-dev.txt`.
- **No logic lives here.** Every cell calls code in `src/`. If a cell starts computing something new, that code belongs in `src/` with a test.
- `autoreload` is on: edits to `src/*.py` apply without restarting the kernel.
- Before committing: *Clear All Outputs*, so git diffs stay readable.

In [ ]:
%load_ext autoreload
%autoreload 2

import subprocess
import sys
from datetime import date, datetime
from pathlib import Path

import openpyxl

ROOT = Path.cwd()
assert (ROOT / "src" / "calculo.py").exists(), "Open this notebook from the repo root"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src import calculo, extractor, loader, motor


def pytest(*args):
    """Run pytest in a subprocess, so it always sees the current code."""
    r = subprocess.run([sys.executable, "-m", "pytest", *args], cwd=ROOT, capture_output=True, text=True)
    print(r.stdout, r.stderr)


def motor_cli(*args):
    """Run the real CLI (python -m src.motor ...) in a subprocess: exactly what you'd type in a terminal."""
    r = subprocess.run([sys.executable, "-m", "src.motor", *args], cwd=ROOT, capture_output=True, text=True)
    print(r.stdout, r.stderr)


def tabla(filas):
    print(f"{'id':>3}  {'fecha':<10}  {'copago':>11}  {'paga benef.':>11}  {'cubre seguro':>12}  {'acumulado':>11}")
    for f in filas:
        print(f"{f.prestacion_id:>3}  {f.fecha}  {f.copago:>11,}  {f.paga_beneficiario:>11,}  {f.cubre_seguro:>12,}  {f.acumulado:>11,}")

## 1. Tests

The same command CI runs. Pass pytest flags as arguments: `-q` quiet, `-v` one line per test, `-k` filter by name, `-x` stop at the first failure.

In [ ]:
pytest("-q")

In [ ]:
pytest("-v", "-k", "escenario")      # the four recalculation scenarios

## 2. Extract: the seed data

A fresh in-memory database with `sql/schema.sql` + `sql/seed.sql`; nothing touches disk.
`extractor` turns rows into typed dataclasses (`Beneficiario`, `ReglaGES`, `Prestacion`) and never writes.

In [ ]:
conn = extractor.conectar(":memory:")
extractor.inicializar(conn)

print(extractor.leer_beneficiario(conn, 1))
print()
REGLAS = extractor.leer_reglas_ges(conn, "GES01")
for r in REGLAS:
    print(f"rule {r.regla_id}: {r.valid_from} → {str(r.valid_to or 'open'):<10}  tope {r.tope_uf} UF")

### What did the system know, and when? (plan D5)

`recibido_en` is when a row arrived. Set `CONOCIDO_HASTA` to see the data as it was known at that moment: with `datetime(2026, 3, 4, 23, 59)`, prestación 7 (received 2026-03-05) disappears. `None` = everything known now.

In [ ]:
CONOCIDO_HASTA = None                       # try: datetime(2026, 3, 4, 23, 59)

print(f"{'id':>3}  {'fecha':<10}  {'recibido_en':<19}  {'código':<6}  {'red GES':<7}  {'copago':>11}")
for x in extractor.leer_prestaciones(conn, 1, CONOCIDO_HASTA):
    print(f"{x.prestacion_id:>3}  {x.fecha_prestacion}  {x.recibido_en}  {x.codigo_ges or '-':<6}  {str(x.en_red_ges):<7}  {x.monto_copago:>11,}")

## 3. Transform: the pure calculation, step by step

`calculo.calcular_ges` does all of this in one call. Here each step is shown on its own, with the seed data from §2, and the last line checks both give the same answer.
No database access in this section: `calculo` receives everything as arguments.

In [ ]:
ben = extractor.leer_beneficiario(conn, 1)
todas = extractor.leer_prestaciones(conn, 1)
valores_uf = extractor.leer_valores_uf(conn)

calificadas = calculo.filtrar_ges(todas, "GES01")
print("1. qualifying (GES01, in network):", [p.prestacion_id for p in calificadas], " ← 3 out of network, 4 not GES")

ventana = calculo.derivar_ventana(calificadas)
print("2. derived window:                ", ventana[0], "→", ventana[1], " ← 12 months from the first copago")

regla = calculo.regla_vigente("GES01", ventana[0], REGLAS)
print("3. rule on the window start (D1): ", f"rule {regla.regla_id}, tope {regla.tope_uf} UF")

dia_uf = calculo.fecha_uf(ventana[0])
ded_uf = calculo.deducible_ges_uf(ben.cotizacion_pactada_uf, regla)
ded_clp = calculo.uf_a_clp(ded_uf, valores_uf[dia_uf])
print("4. deductible:                    ", f"min(29 × {ben.cotizacion_pactada_uf}, {regla.tope_uf}) = {ded_uf} UF × UF({dia_uf}) = {ded_clp:,} CLP")

filas = calculo.desglose(calculo.en_ventana(calificadas, ventana), ded_clp)
print("5. split:                          crossed on", calculo.fecha_cruce(filas, ded_clp), " ← 9 is after the window end\n")
tabla(filas)

assert calculo.calcular_ges(ben, todas, REGLAS, "GES01", valores_uf).filas == filas

### What-if: before the late arrival was known

The same pure call without prestación 7: the crossing moves from 2025-11-20 to 2026-02-10.

In [ ]:
sin_7 = calculo.calcular_ges(ben, [p for p in todas if p.prestacion_id != 7], REGLAS, "GES01", valores_uf)
print("crossed on", sin_7.fecha_cruce)
tabla(sin_7.filas)

## 4. Load: the full pipeline

`motor.calcular` runs extract → transform → load and saves the result with its metadata (rule version, window, what the run knew, code version, data warnings).
`loader.reporte_texto` prints it back **from the database**, not from memory. The second call finds nothing new: `sin_cambios`, nothing saved.

In [ ]:
AHORA = datetime(2026, 9, 27, 12, 0)

db = extractor.conectar(":memory:")
extractor.inicializar(db)
cid, motivo = motor.calcular(db, 1, "GES", AHORA, version=motor.version_codigo())
print(loader.reporte_texto(extractor.leer_resultado(db, cid)))
print()
print("second run:", motor.calcular(db, 1, "GES", datetime(2026, 9, 28)))

## 5. Recalculation: the four scenarios (spec §6)

One database, one beneficiary, a timeline of events. Each run saves a new version with its reason, or says `sin_cambios`. Nothing is overwritten.

| Reason | What changed | Rule used |
|---|---|---|
| `datos_tardios` | a late prestación inside the window | **the rule of the period**, not today's |
| `ventana_desplazada` | a late prestación before the window start | the rule on the new start |
| `cambio_normativo` | a rule version loaded (here retroactively) | the new one |
| `sin_cambios` | nothing | — nothing saved |

In [ ]:
h = extractor.conectar(":memory:")
extractor.inicializar(h)


def paso(ahora, nota):
    cid, motivo = motor.calcular(h, 1, "GES", ahora)
    print(f"{ahora:%Y-%m-%d}  {nota:<52} → {motivo}")


paso(datetime(2026, 3, 4, 23, 59), "first run (prestación 7 not received yet)")
paso(datetime(2026, 3, 6), "prestación 7 arrived on 2026-03-05")
paso(datetime(2026, 3, 7), "nothing new")

loader.cargar_regla_ges(h, "GES01", date(2025, 3, 1), 29, 125.0, "Decreto (demo)", "notebook", datetime(2026, 4, 1))
paso(datetime(2026, 4, 2), "retroactive rule: 125 UF from 2025-03-01")

tardia = {"beneficiario_id": 1, "fecha_prestacion": date(2025, 2, 5), "codigo_ges": "GES01",
          "en_red_ges": True, "en_red_caec": False, "monto_arancel": 2_500_000, "monto_copago": 500_000}
loader.insertar_prestaciones(h, [tardia], recibido_en=datetime(2026, 5, 1, 9, 0))
paso(datetime(2026, 5, 2), "late prestación dated 2025-02-05 (before the window)")

print()
print(loader.reporte_historial(extractor.leer_historial(h, 1, "GES")))

In [ ]:
# The rule set after the retroactive load: version 1 was split, nothing was deleted.
for r in extractor.leer_reglas_ges(h, "GES01"):
    print(f"rule {r.regla_id}: {r.valid_from} → {str(r.valid_to or 'open'):<10}  tope {r.tope_uf} UF")

## 6. Excel ingestion

Builds a sample export in format **B** (insurer: all text, `DD-MM-YYYY` dates, `$ 1.234.567` amounts, 1/0 flags) with one good row and three bad ones, then runs the real CLI on `data/runner.db` (git-ignored, recreated here).
Nothing is dropped silently: each rejected row is listed with its reason. Edit the rows and re-run to experiment. Column layouts: `motor.FORMATOS`.

In [ ]:
MUESTRA = ROOT / "data" / "muestra_formato_b.xlsx"          # git-ignored
MUESTRA.parent.mkdir(exist_ok=True)
libro = openpyxl.Workbook()
hoja = libro.active
hoja.append(list(motor.FORMATOS["B"].values()))
for fila in [
    ["1", "01-12-2025", "GES01", "1", "0", "$ 1.000.000", "$ 200.000"],   # ok, inside beneficiary 1's window
    ["1", "31-02-2025", "GES01", "1", "0", "$ 100", "$ 20"],              # impossible date
    ["1", "10-03-2025", "GES01", "1", "0", "5.000.000", "1.000.000"],     # same as seed prestación 1
    ["77", "01-12-2025", "GES01", "1", "0", "100", "20"],                  # unknown beneficiary
]:
    hoja.append(fila)
libro.save(MUESTRA)

(ROOT / "data" / "runner.db").unlink(missing_ok=True)
motor_cli("init", "--db", "data/runner.db")
motor_cli("recalcular-todos", "--db", "data/runner.db", "--tipo", "GES", "--ahora", "2026-09-27T12:00")
motor_cli("ingerir", "--db", "data/runner.db", "--archivo", "data/muestra_formato_b.xlsx", "--formato", "B", "--ahora", "2026-09-28T09:00")
motor_cli("recalcular-todos", "--db", "data/runner.db", "--tipo", "GES", "--ahora", "2026-09-28T10:00")
motor_cli("ejecuciones", "--db", "data/runner.db")

## 7. Your database: operate the engine

Everything above uses throwaway databases. This section works on **`data/motor.db`**, which is kept between sessions: this is the notebook as your CLI.
Each cell is one command, exactly what you'd type in a terminal (`python -m src.motor ...`). Every command that writes is logged; see the last cell.
Change the values and run the cell you need. To start over, delete `data/motor.db`.

In [ ]:
DB = "data/motor.db"

if (ROOT / DB).exists():
    print(f"{DB} exists: using it")
else:
    motor_cli("init", "--db", DB)

In [ ]:
# What the database holds for a beneficiary: details, prestaciones (and when each arrived), current results
motor_cli("ver", "--db", DB, "--beneficiario", "1")
# option: "--conocido-hasta", "2026-03-04T23:59"   only prestaciones received by then

In [ ]:
# Calculate one beneficiary (saves a new version only if something changed)
motor_cli("calcular", "--db", DB, "--beneficiario", "1", "--tipo", "GES")
# options: "--ahora", "2026-09-27T12:00"   "--codigo", "GES01"

In [ ]:
# What-if, saves nothing: what the engine would say with only the data received by a given moment
motor_cli("calcular", "--db", DB, "--beneficiario", "1", "--tipo", "GES", "--simular",
          "--conocido-hasta", "2026-03-04T23:59")

In [ ]:
# Calculate everyone
motor_cli("recalcular-todos", "--db", DB, "--tipo", "GES")

In [ ]:
# Load a rule version (future or retroactive). Uncomment to run: it changes the rule set for good.
# motor_cli("cargar-regla-ges", "--db", DB, "--codigo", "GES01", "--desde", "2027-01-01",
#           "--tope-uf", "135", "--fuente", "Decreto X")

In [ ]:
# Load beneficiaries and prestaciones from Excel. The example file is created once and never overwritten:
# edit it (add beneficiaries, prestaciones), then re-run this cell. Rows already loaded come back as duplicates.
ARCHIVO = "data/carga_ejemplo.xlsx"
if not (ROOT / ARCHIVO).exists():
    motor_cli("plantilla", "--archivo", ARCHIVO)
motor_cli("ingerir", "--db", DB, "--archivo", ARCHIVO, "--formato", "A")
motor_cli("recalcular-todos", "--db", DB, "--tipo", "GES")

In [ ]:
# Every saved version for a beneficiary, and why it changed
motor_cli("historial", "--db", DB, "--beneficiario", "1", "--tipo", "GES")

In [ ]:
# The run log: who ran what, when, with which code, and the outcome
motor_cli("ejecuciones", "--db", DB, "--ultimas", "20")

In [ ]:
# Help for any command
motor_cli("--help")
# motor_cli("calcular", "--help")

## Next

Phase 6: CI (GitHub Actions running the tests on every push) and the final README. CAEC (phase 2) and loading rules from Excel (S14) are deferred.